# distilroute — fine-tune the transformer students on Colab's free GPU

Runtime → Change runtime type → **T4 GPU**. Everything here is `scripts/finetune.py` from the
repo; this notebook only fetches the code, runs the configs, and hands back the outputs.
Each run writes `results/<name>.json` + `_test_probs.npz` (the shared run contract) and, with
`--export-onnx`, `models/<name>/model.int8.onnx` — copy those into the laptop checkout and
`scripts/evaluate.py` picks them up.

In [ ]:
# 1. Get the code. Once the repo is public, set REPO_URL; until then upload a zip of the
#    checkout (without .venv/ and data/raw/ — raw data is re-downloaded below).
REPO_URL = ""  # e.g. "https://github.com/<user>/distilroute.git"

import os, shutil
if REPO_URL:
    !git clone -q $REPO_URL distilroute
else:
    from google.colab import files
    up = files.upload()  # pick distilroute.zip
    !unzip -q -o {list(up)[0]}
    if not os.path.isdir("distilroute"):
        raise SystemExit("zip must unpack to a distilroute/ folder")
%cd distilroute
!ls

In [ ]:
# 2. Dependencies (Colab already ships torch + transformers) and the raw data.
!pip install -q -r requirements.txt onnx onnxruntime
!python scripts/download_data.py
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. The runs. Roughly 4 min per DistilBERT run on a T4 (10k rows x 4 epochs); TinyBERT ~1 min.
#    `teacher` runs need data/labels/train.jsonl in the repo (roadmap 1.6); skip them until then.
RUNS = [
    "--model distilbert --labels gold    --export-onnx",
    "--model minilm     --labels gold    --export-onnx",
    "--model tinybert   --labels gold    --export-onnx",
    "--model distilbert --labels teacher --export-onnx",
    "--model distilbert --labels teacher --export-onnx --soft",
    "--model minilm     --labels teacher --export-onnx",
    "--model tinybert   --labels teacher --export-onnx",
]
have_teacher = os.path.exists("data/labels/train.jsonl")
for args in RUNS:
    if "teacher" in args and not have_teacher:
        print("skip (no train labels yet):", args)
        continue
    print()
    print("===", args)
    !python scripts/finetune.py $args

In [ ]:
# 4. Bundle everything the laptop needs: metrics + test probabilities + int8 ONNX models
#    (fp32 .onnx left out — 260 MB for DistilBERT and never served).
!rm -f distilroute_runs.zip
!zip -q -r distilroute_runs.zip results models -x "models/*/model.onnx" "models/cache/*"
!ls -lh distilroute_runs.zip
from google.colab import files
files.download("distilroute_runs.zip")

Back on the laptop: unzip into the checkout (it only adds `results/*ft*` and `models/*`),
then `python scripts/evaluate.py` and `python scripts/bench_latency.py` (roadmap 3.2) to
re-measure latency on the laptop's CPU — the `p50_ms` in the JSON is Colab's CPU.